# ViFinQA — Kaggle GPU: dense retrieval and grounded generation
Bật GPU và Internet (hoặc attach model/dataset). Chạy smoke 5 câu trước khi bỏ `--limit`. Notebook không giả định loại hay số GPU.

In [ ]:
# Immutable revisions verified before the competition cutoff. Edit only the two run flags below.
import os

os.environ["VIFINQA_MODEL_REVISION"] = "8e8ed243bbe6f9a5aff549a0924562fc719b2b8a"
os.environ["VIFINQA_DENSE_REVISION"] = "5617a9f61b028005a4858fdac845db406aefb181"
os.environ["VIFINQA_FINAL_RUN"] = "0"
os.environ["VIFINQA_RUN_FULL"] = "0"
os.environ["VIFINQA_TP"] = "1"
print(
    "final/full/tp:",
    os.environ["VIFINQA_FINAL_RUN"],
    os.environ["VIFINQA_RUN_FULL"],
    os.environ["VIFINQA_TP"],
)

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import requests
import torch

print(
    "torch", torch.__version__, "cuda", torch.cuda.is_available(), "gpus", torch.cuda.device_count()
)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / 2**30, 1), "GiB")
assert torch.cuda.is_available(), "Enable a GPU accelerator before continuing."
FINAL_RUN = os.environ.get("VIFINQA_FINAL_RUN") == "1"
RUN_FULL = os.environ.get("VIFINQA_RUN_FULL") == "1"
MODEL_REVISION = os.environ.get("VIFINQA_MODEL_REVISION")
DENSE_REVISION = os.environ.get("VIFINQA_DENSE_REVISION")
if FINAL_RUN:
    assert MODEL_REVISION and DENSE_REVISION, "Final run requires pre-cutoff model commit SHAs."

In [ ]:
# Prefer an attached Kaggle Dataset containing the current repo; otherwise clone.
GIT_URL = "https://github.com/ThanhDatVN/AI-Financial-Data-Assistant.git"
PROJECT = Path("/kaggle/working/AI-Financial-Data-Assistant")
ATTACHED_REPO = Path("/kaggle/input/ai-financial-data-assistant")
if not PROJECT.exists() and ATTACHED_REPO.exists():
    shutil.copytree(ATTACHED_REPO, PROJECT)
elif not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "sentence-transformers==3.3.1",
        "faiss-cpu==1.9.0.post1",
        "vllm==0.25.1",
    ],
    check=True,
)
os.chdir(PROJECT)
PROJECT_SHA = (
    subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    if (PROJECT / ".git").exists()
    else "attached-archive-no-git-sha"
)
if FINAL_RUN:
    assert (
        PROJECT_SHA != "attached-archive-no-git-sha"
    ), "Final run must use a Git checkout with a recorded SHA."
print("project revision:", PROJECT_SHA)
RUNTIME_LOG = Path("/kaggle/working/runtime_environment.txt")
RUNTIME_LOG.write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True),
    encoding="utf-8",
)

In [ ]:
# Point these to attached datasets/artefacts when available.
DATA_ROOT = Path("/kaggle/input/vifinqa/ViFinQA")
ARTIFACT_INPUT = Path("/kaggle/input/vifinqa-artifacts")
if not DATA_ROOT.exists():
    DATA_ROOT = PROJECT / "data/raw/ViFinQA"
MANIFEST = ARTIFACT_INPUT / "data/processed/table_manifest.jsonl"
if not MANIFEST.exists():
    MANIFEST = PROJECT / "data/processed/table_manifest.jsonl"
BM25 = ARTIFACT_INPUT / "data/index/bm25"
if not BM25.exists():
    BM25 = PROJECT / "data/index/bm25"
assert DATA_ROOT.exists(), "Attach ViFinQA or download it before this cell."
assert MANIFEST.exists(), "Run the Colab preparation notebook or build the manifest here."
assert MANIFEST.with_suffix(".parquet").exists(), "The generation stage needs the Parquet manifest."
assert BM25.exists(), "Attach or build the BM25 index."


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


expected_hashes = {
    MANIFEST: "ced1d671d6a71c299fea02d7d12b86b596b430a2714ab9aeaa4b338fe012fac1",
    MANIFEST.with_suffix(
        ".parquet"
    ): "060bd26eff14d30ce70b3ba7b00af509be6100b58ddb5a0fe970afa0ef69e29d",
    MANIFEST.with_suffix(
        ".metadata.json"
    ): "bb338c8a01381241a915517fe774b045cc53213eddd4194f645473616c188e12",
    BM25 / "data.csc.index.npy": "3f2d7292960e8fda6ca5a1f09d10f692259629ac66cc61703b275b927d5cd683",
    BM25
    / "indices.csc.index.npy": "6f5ac7fa7be96f946eced6dcfa7eddeb63493d317f13ef4249bc287e3d993991",
    BM25
    / "indptr.csc.index.npy": "6c136f30e633641b65b26ba2341a0a0aedbba2932aa1d51b4b48e953fbf247eb",
    BM25 / "params.index.json": "b42a70b508494aa5bb40ef81323a35b4b4ed5afbb7984ed224bb700197a01e7c",
    BM25 / "records.jsonl": "1b8ebe896b92e77c71e5ebadb2e519b377932e1b5a9665080e457fce12daf40f",
    BM25 / "vocab.index.json": "3c5482d9e193cb4240e329de6406779ee5a67f1b4f570bde0d4c397aa297f5aa",
}
for path, expected in expected_hashes.items():
    assert path.exists(), f"Missing frozen artefact: {path}"
    actual = sha256(path)
    assert actual == expected, f"SHA-256 mismatch for {path}: {actual}"
question_count = sum(
    1 for line in (DATA_ROOT / "questions/questions.jsonl").open(encoding="utf-8") if line.strip()
)
assert question_count == 1012, f"Expected 1,012 questions, found {question_count}"
print("inputs verified:", DATA_ROOT, MANIFEST, BM25, "questions=", question_count)

In [ ]:
DENSE = Path("/kaggle/working/artifacts/bge_m3")
if not (DENSE / "index.faiss").exists():
    dense_cmd = [
        sys.executable,
        "scripts/22_build_dense.py",
        "--manifest",
        str(MANIFEST),
        "--output",
        str(DENSE),
        "--device",
        "cuda",
        "--batch-size",
        "16",
    ]
    if DENSE_REVISION:
        dense_cmd += ["--model-revision", DENSE_REVISION]
    if FINAL_RUN:
        dense_cmd += ["--final-run"]
    subprocess.run(dense_cmd, check=True)
dense_config = json.loads((DENSE / "config.json").read_text(encoding="utf-8"))
assert (
    dense_config.get("tables") == 146246
), f"Dense index must contain 146,246 tables: {dense_config}"
if FINAL_RUN:
    assert (
        dense_config.get("model_revision") == DENSE_REVISION
    ), "Existing dense index was not built with the approved revision."
RETRIEVAL = Path("/kaggle/working/artifacts/retrieval_hybrid.jsonl")
subprocess.run(
    [
        sys.executable,
        "scripts/30_retrieve_questions.py",
        "--questions",
        str(DATA_ROOT / "questions/questions.jsonl"),
        "--companies",
        str(DATA_ROOT / "code_stock.csv"),
        "--bm25",
        str(BM25),
        "--dense",
        str(DENSE),
        "--output",
        str(RETRIEVAL),
        "--candidate-k",
        "2000",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "scripts/32_validate_retrieval.py",
        "--questions",
        str(DATA_ROOT / "questions/questions.jsonl"),
        "--manifest",
        str(MANIFEST.with_suffix(".parquet")),
        "--retrieval",
        str(RETRIEVAL),
        "--output",
        "/kaggle/working/artifacts/retrieval_hybrid_qc.json",
    ],
    check=True,
)
retrieval_qc_path = Path("/kaggle/working/artifacts/retrieval_hybrid_qc.json")
retrieval_qc = json.loads(retrieval_qc_path.read_text(encoding="utf-8"))
assert retrieval_qc.get("passed") is True and retrieval_qc.get("rows") == 1012, retrieval_qc
print("hybrid retrieval verified:", retrieval_qc_path, retrieval_qc["retrieval_sha256"])

In [ ]:
# Start one deterministic OpenAI-compatible vLLM server.
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"
TP = int(os.environ.get("VIFINQA_TP", "1"))
assert 1 <= TP <= torch.cuda.device_count(), "VIFINQA_TP must be between 1 and the GPU count."
server_log = open("/kaggle/working/vllm.log", "w")  # noqa: SIM115
serve_cmd = [
    "vllm",
    "serve",
    MODEL,
    "--host",
    "127.0.0.1",
    "--port",
    "8000",
    "--tensor-parallel-size",
    str(TP),
    "--max-model-len",
    "8192",
    "--gpu-memory-utilization",
    "0.90",
    "--seed",
    "20260802",
]
if MODEL_REVISION:
    serve_cmd += ["--revision", MODEL_REVISION]
server = subprocess.Popen(serve_cmd, stdout=server_log, stderr=subprocess.STDOUT)

for _ in range(120):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).ok:
            break
    except requests.RequestException:
        pass
    if server.poll() is not None:
        raise RuntimeError(Path("/kaggle/working/vllm.log").read_text()[-4000:])
    time.sleep(5)
else:
    raise TimeoutError("vLLM did not become healthy; inspect /kaggle/working/vllm.log")
print("vLLM ready")

In [ ]:
# Smoke run. Inspect errors.jsonl and predictions before running all 1,012 questions.
GEN = Path("/kaggle/working/artifacts/generation")
smoke_cmd = [
    sys.executable,
    "scripts/50_generate_programs.py",
    "--retrieval",
    str(RETRIEVAL),
    "--manifest",
    str(MANIFEST.with_suffix(".parquet")),
    "--data-root",
    str(DATA_ROOT),
    "--output",
    str(GEN),
    "--model",
    MODEL,
    "--limit",
    "5",
]
if MODEL_REVISION:
    smoke_cmd += ["--model-revision", MODEL_REVISION]
subprocess.run(smoke_cmd, check=True)
smoke_errors = []
if (GEN / "errors.jsonl").exists():
    smoke_errors = [
        json.loads(line) for line in (GEN / "errors.jsonl").read_text().splitlines() if line
    ]
smoke_predictions = json.loads((GEN / "submission.json").read_text(encoding="utf-8"))
smoke_traces = []
if (GEN / "program_traces.jsonl").exists():
    smoke_traces = [
        json.loads(line)
        for line in (GEN / "program_traces.jsonl").read_text(encoding="utf-8").splitlines()
        if line
    ]
print(
    "smoke predictions/errors/traces:", len(smoke_predictions), len(smoke_errors), len(smoke_traces)
)
print(json.dumps(smoke_predictions[:2], ensure_ascii=False, indent=2)[:4000])
assert not smoke_errors, "Smoke recorded errors; inspect errors.jsonl before continuing."
assert (
    len(smoke_predictions) == 5 and len(smoke_traces) == 5
), "Smoke must produce 5 predictions and 5 traces."

In [ ]:
# Full resume-safe generation. Enable RUN_FULL in config, rerun config, then run this cell.
RUN_FULL = os.environ.get("VIFINQA_RUN_FULL") == "1"
FINAL_RUN = os.environ.get("VIFINQA_FINAL_RUN") == "1"
assert RUN_FULL, "Set VIFINQA_RUN_FULL=1 only after inspecting smoke output."
full_cmd = [
    sys.executable,
    "scripts/50_generate_programs.py",
    "--retrieval",
    str(RETRIEVAL),
    "--manifest",
    str(MANIFEST.with_suffix(".parquet")),
    "--data-root",
    str(DATA_ROOT),
    "--output",
    str(GEN),
    "--model",
    MODEL,
]
if MODEL_REVISION:
    full_cmd += ["--model-revision", MODEL_REVISION]
if FINAL_RUN:
    full_cmd += ["--final-run"]
subprocess.run(full_cmd, check=True)
subprocess.run(
    [
        sys.executable,
        "scripts/40_validate_submission.py",
        str(GEN / "submission.json"),
        "--questions",
        str(DATA_ROOT / "questions/questions.jsonl"),
        "--evidence-root",
        str(GEN),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "scripts/41_package_submission.py",
        str(GEN / "submission.json"),
        "/kaggle/working/submission.zip",
        "--questions",
        str(DATA_ROOT / "questions/questions.jsonl"),
        "--evidence-root",
        str(GEN),
    ],
    check=True,
)
submission_zip = Path("/kaggle/working/submission.zip")
final_predictions = json.loads((GEN / "submission.json").read_text(encoding="utf-8"))
final_errors = []
if (GEN / "errors.jsonl").exists():
    final_errors = [
        line for line in (GEN / "errors.jsonl").read_text(encoding="utf-8").splitlines() if line
    ]
assert len(final_predictions) == 1012 and not final_errors
print(submission_zip, submission_zip.stat().st_size, sha256(submission_zip))